# X86 PT2E INT8 quantization: a visual walkthrough
This notebook explains the default Post training **static** `X86InductorQuantizer` path with text graphs. It uses a tiny Conv2d example so you can inspect real graph nodes. `torch.compile` is **not run**: your MSVC 14.28 toolchain does not accept the C++20 flag produced by Inductor. The toy graph is illustrative; your detector has many branches and not all operators are quantized.

Pipeline: `FP32 model → export → x86 quantizer recipe → prepare → calibrate → convert → (optional) Inductor lowering`.


## 0. Start with a floating-point model
```text
input FP32 ──> Conv weights FP32 ──> ReLU ──> output FP32
```
An activation is an intermediate tensor; a weight is a learned tensor. A convolution weight tensor can contain thousands of numbers but counts as one tensor in a dtype `Counter`. Quantization aims to represent *selected* values with 8 bits using a scale and zero-point. Conceptually, `q = clamp(round(x / scale) + zero_point)` and `x_approx = (q - zero_point) * scale`. The rounding makes the approximation lossy. The default x86 recipe uses 8-bit activations and weights; it is static unless `is_dynamic=True` is explicitly requested.


In [1]:
import torch
import torch.nn as nn
from collections import Counter
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer, get_default_x86_inductor_quantization_config
)

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x))

float_model = TinyCNN().eval()
example_inputs = (torch.randn(1, 3, 32, 32),)
print('Parameter tensors:', len(list(float_model.parameters())))
print('Scalar parameters:', sum(p.numel() for p in float_model.parameters()))


W0930 10:06:43.356000 9292 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Parameter tensors: 2
Scalar parameters: 224


## 1. Export the computation
```text
Python model:  self.relu(self.conv(x))
                       │ export with example input
                       ▼
FX graph:  input → ATen convolution → ATen relu → output
```
Export records a lower-level operation graph that the quantizer can analyze. One source-code 'layer' need not map to exactly one graph node. The example input helps capture this execution path; your detector used `(1, 3, 640, 640)`.


In [2]:
with torch.no_grad():
    exported_model = torch.export.export(float_model, example_inputs).module()
print(exported_model.graph)


graph():
    %conv_weight : [num_users=1] = get_attr[target=conv.weight]
    %conv_bias : [num_users=1] = get_attr[target=conv.bias]
    %x : [num_users=2] = placeholder[target=x]
    %_guards_fn : [num_users=0] = call_module[target=_guards_fn](args = (%x,), kwargs = {})
    %conv2d : [num_users=1] = call_function[target=torch.ops.aten.conv2d.default](args = (%x, %conv_weight, %conv_bias, [1, 1], [1, 1]), kwargs = {})
    %relu : [num_users=1] = call_function[target=torch.ops.aten.relu.default](args = (%conv2d,), kwargs = {})
    return (relu,)


## 2. Choose the x86 quantization recipe
```text
FX graph ──> X86InductorQuantizer(default static config)
                   │
                   └─ annotates eligible tensor inputs/outputs
                      and weight paths for 8-bit quantization
```
`set_global(...)` supplies a default policy across supported patterns; it does **not** mean every operation or every parameter becomes INT8. Other operations can remain FP32. The policy is aimed at later x86 Inductor lowering.


In [3]:
quantizer = X86InductorQuantizer()
quantizer.set_global(get_default_x86_inductor_quantization_config())
print('Using default static x86 quantization config')


Using default static x86 quantization config


## 3. Prepare: insert observers
```text
activation ──> [observer: watch range] ──> convolution ──> [observer] ──> next
weight     ──> [observer: inspect range] ────────┘
```
`prepare_pt2e` places observers where the backend recipe calls for them; it can also fold BatchNorm into a preceding convolution. Observers measure values without yet making an optimized INT8 kernel. Exact placements vary with the graph.


In [4]:
prepared_model = prepare_pt2e(exported_model, quantizer)
print('Observer-like graph targets:')
for node in prepared_model.graph.nodes:
    if 'activation_post_process' in str(node.target):
        print(node.name, node.target)


Observer-like graph targets:
activation_post_process_1 activation_post_process_1
activation_post_process_0 activation_post_process_0


## 4. Calibrate on representative FP32 inputs
```text
sample images ──> prepared graph ──> observers collect ranges
                                   ↓
                              scale / zero-point
```
This is **post-training static quantization**: observers determine conversion parameters from calibration data rather than estimating them afresh for each future input. The random toy inputs below demonstrate the API only; calibrate your detector with representative real images at its exported batch size and shape. Calibration is not training and does not update learned convolution weights.


In [5]:
with torch.no_grad():
    for _ in range(8):
        prepared_model(torch.randn_like(example_inputs[0]))
print('Toy calibration completed')


Toy calibration completed


## 5. Convert: add explicit quantization boundaries
```text
FP32 activation ── Q ── DQ ──┐
                             ├── convolution ── Q ── DQ ── next
FP32 weight ───── Q ── DQ ──┘
```
**Q** maps values to an integer representation using calibration parameters; **DQ** maps them back approximately. This diagram illustrates a possible reference-style pattern, **not** a promise that the tiny example has exactly these nodes or that eager convolution itself runs INT8. Q/DQ marks quantization of graph tensors/edges, not one pair per original layer. A weight path, branching users, and shared boundaries can change counts. `convert_pt2e` replaces observers with quantization/dequantization graph operations.


In [6]:
converted_model = convert_pt2e(prepared_model)
quant_nodes = [n for n in converted_model.graph.nodes if 'quantiz' in str(n.target).lower()]
print('Q/DQ-related node targets:', Counter(str(n.target) for n in quant_nodes))
print('Matched graph node count:', len(quant_nodes))
print('Visible parameter dtypes:', Counter(p.dtype for p in converted_model.parameters()))


Q/DQ-related node targets: Counter({'quantized_decomposed.dequantize_per_channel.default': 1, 'quantized_decomposed.quantize_per_tensor.default': 1, 'quantized_decomposed.dequantize_per_tensor.default': 1})
Matched graph node count: 3
Visible parameter dtypes: Counter({torch.float32: 2})


## 6. Link Q/DQ to operations
```text
upstream node ──> Q/DQ node ──> downstream node(s)
      node.args             node.users

Original FP32:
input 1.23 ───────────────→ FP32 convolution ──→ prediction

Converted, uncompiled:
input 1.23 ── Q → integer ── DQ → approximately 1.23
                                      │
                                      └──→ FP32 convolution ──→ prediction
```
Inspect edges instead of dividing a Q/DQ count by two. Q/DQ may occur on an activation or on a weight, and there may be multiple consumers. Exported `nn_module_stack` metadata may identify an original module, but is not guaranteed to survive every transformation.


In [7]:
from torch.fx import Node
for node in quant_nodes[:12]:
    incoming = [(a.name, str(a.target)) for a in node.args if isinstance(a, Node)]
    outgoing = [(u.name, str(u.target)) for u in node.users]
    print(node.name, str(node.target))
    print('  input from:', incoming)
    print('  output to:', outgoing)
print('Original module metadata example:')
for node in converted_model.graph.nodes:
    if 'convolution' in str(node.target).lower():
        print(node.name, node.meta.get('nn_module_stack'))


dequantize_per_channel_default quantized_decomposed.dequantize_per_channel.default
  input from: [('quantize_per_channel_default', '_frozen_param0'), ('_scale_0', '_scale_0'), ('_zero_point_0', '_zero_point_0')]
  output to: [('conv2d', 'aten.conv2d.default')]
quantize_per_tensor_default quantized_decomposed.quantize_per_tensor.default
  input from: [('x', 'x')]
  output to: [('dequantize_per_tensor_default', 'quantized_decomposed.dequantize_per_tensor.default')]
dequantize_per_tensor_default quantized_decomposed.dequantize_per_tensor.default
  input from: [('quantize_per_tensor_default', 'quantized_decomposed.quantize_per_tensor.default')]
  output to: [('conv2d', 'aten.conv2d.default')]
Original module metadata example:


## 7. Intended Inductor lowering (NOT run here)
```text
converted:   activation → Q → DQ ──┐
                                  ├→ Conv → Q → DQ → next
             weight → Q → DQ ──────┘
                         │ recognize eligible pattern
                         ▼
lowered:     optimized quantized convolution + necessary boundary conversions
```
`torch.compile(converted_model)` normally invokes Inductor to recognize/fuse eligible patterns and select optimized kernels. This step requires a compatible C++ toolchain for your Windows CPU workload. **Do not run it in this notebook with MSVC 14.28**. A converted model run without this stage can be slower because Q/DQ incurs work without the intended fusion. Neither a count of Q/DQ nodes nor visible FP32 parameter dtypes proves which optimized kernels execute.


## 8. Interpret your detector results
```text
Your detector: FP32 → export (1,3,640,640) → prepare → calibrate → convert
                                     └─ 389 name matches in converted graph
                                     └─ no successful Inductor lowering
Observed eager latency: FP32 42.15 ms; converted 70.13 ms; ratio 0.60×
```
389 is the count of nodes whose targets match the substring `quantiz`, including Q and DQ targets; it is **not** a layer count, pair count, or optimized-kernel count. Your latency comparison describes this *uncompiled* configuration only.


## Documentation
- PyTorch/torchao: https://docs.pytorch.org/ao/0.17/pt2e_quantization/pt2e_quant_x86_inductor.html
- PT2E overview: https://docs.pytorch.org/ao/stable/pt2e_quantization/index.html
- PyTorch FX graph/node concepts: https://docs.pytorch.org/docs/stable/fx.html

Notebook assumes PyTorch and torchao with the imports shown; API behavior can vary by installed version.
